# B4 – Oblikovanje značilk

**Samostojno delo · Telekom**


## Priprava

Notebook je samostojen in deluje lokalno ter v Google Colabu. Celice izvajaj od
zgoraj navzdol s **Shift + Enter**. Ob menjavi stolpcev ponovi ustrezne naslednje celice.

Prva celica poišče `utils/orodja.py`; v Colabu po potrebi prenese repozitorij in namesti
knjižnice iz `requirements.txt`. Lokalno jih enkrat namestimo po navodilih v README.
Druga celica pokliče skupni `poskrbi_za_podatke()` in vrne mapo `DATA`.
Funkcija uporabi pripravljene ali že prenesene podatke; manjkajoče prenese s Kaggla.
CSV-je nato beremo iz `DATA`, ne glede na mapo, v kateri je zvezek odprt.
Izvornih vrednosti ne popravljamo in rezultatov vaj ne zapisujemo samodejno v datoteke.

Najprej oblikujemo ponovljive značilke, njihovo uporabnost pa pozneje preverimo z validacijo modela.


In [ ]:
#@title Nastavitev okolja (kliknite za prikaz kode)
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/valira-ai/ui-za-poslovne-aplikacije.git"

if "google.colab" in sys.modules:
    KOREN = Path("/content/ui-za-poslovne-aplikacije")
    if not (KOREN / "utils/orodja.py").is_file():
        if KOREN.exists():
            raise FileNotFoundError(f"Mapa {KOREN} že obstaja, vendar v njej ni utils/orodja.py.")
        subprocess.run(["git", "clone", "-q", REPO_URL, str(KOREN)], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(KOREN / "requirements.txt")],
        check=True,
    )
else:
    KOREN = Path.cwd().resolve()
    while not (KOREN / "utils/orodja.py").is_file() and KOREN != KOREN.parent:
        KOREN = KOREN.parent
    if not (KOREN / "utils/orodja.py").is_file():
        raise FileNotFoundError("Zaženite zvezek v prenesenem repozitoriju ali njegovi podmapi (manjka utils/orodja.py).")

if str(KOREN) not in sys.path:
    sys.path.insert(0, str(KOREN))
print("Gradivo je v:", KOREN)


### Skupni podatkovni vhod
`poskrbi_za_podatke()` iz `utils/orodja.py` pripravi skupno mapo podatkov in jo vrne kot `DATA`.

V tem sklopu beremo `DATA / "telekom_izvirni.csv"`: vseh 100 stolpcev z izvirnimi imeni.
Uvod in vizualizacija uporabljata 22-stolpčni slovenski izbor `telekom.csv`;
obe tabeli izhajata iz istih strank in ju pripravi isti podatkovni vhod.


In [ ]:
from utils.orodja import poskrbi_za_podatke
DATA = poskrbi_za_podatke()


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", 15)
pd.set_option("display.max_colwidth", 35)
plt.rcParams.update({"figure.figsize": (9, 4), "font.size": 11})


## 1. Kaj predstavlja vrstica in kaj smemo iz podatkov izpeljati?

Ena vrstica je zapis stranke, `Customer_ID` ključ in `churn` cilj. Nimamo dnevnika
klicev ali datumov posameznih dogodkov, zato ne ustvarjamo domnevnega števila
stikov v zadnjih 30 dneh ali izmišljenega datuma sklenitve naročnine.

| Značilka | Začetni vhod | Druge možnosti za samostojen preizkus |
|---|---|---|
| Razmerje agregatov | `avg3mou` / `avg6mou` | `avg3rev` / `avg6rev` |
| Drugo smiselno razmerje | — | `rev_Mean` / `mou_Mean`, ob preverjanju enot in fiksnih obračunov |
| Kazalnik stika | `custcare_Mean > 0` | Ohrani NaN, kadar vhod ni znan |
| Nominalna kategorija | `area` | `new_cell`, `hnd_webcap`, `refurb_new` |
| Urejeni razredi | Izpeljani iz `months` | Prilagodi smiselne meje in njihov izrecni vrstni red |

Razmerje 3m/6m ni dokaz rasti med dvema zaporednima obdobjema: okni se lahko
prekrivata. Pred napovedovanjem churn je treba potrditi definicije in dostopnost
agregatov v času napovedi. Razmerje prihodka in minut tudi ni samodejno cena minute.


In [ ]:
df = pd.read_csv(DATA / "telekom_izvirni.csv")
assert df["Customer_ID"].is_unique
kljuci = df["Customer_ID"].copy()
cilj = df["churn"].copy()  # Ostane ločeno od vhodnih značilk.
print(f"Strank: {len(df):,}; ključ je enoličen.")


## 2. Razmerje z zaščito pred ničelnim imenovalcem

Spremeni `stevec` in `imenovalec`, nato ponovno izvedi naslednje celice. Pri 0 ali
neznanem imenovalcu razmerje ostane NaN, ne 0 in ne neskončnost. Ločen kazalnik
pove, ali je bil imenovalec **znano** ničeln; pri neznanem vhodu ostane neznan.

Pomen ničle in združljivost enot preverimo za izbrani par. Ne določamo, da je
vsako negativno razmerje napaka, ne da bi poznali definicijo vhodov.


In [ ]:
stevec = "avg3mou"       # Alternativa: "avg3rev".
imenovalec = "avg6mou"   # Ob zgornji alternativi: "avg6rev".
kategorija = "area"     # Poskusi še "new_cell", "hnd_webcap" ali "refurb_new".

znacilke = df[["months", stevec, imenovalec, "custcare_Mean", kategorija]].copy()
znacilke["razmerje_agregatov"] = df[stevec] / df[imenovalec].replace(0, np.nan)
znacilke["imenovalec_je_nic"] = df[imenovalec].eq(0).astype(float).where(df[imenovalec].notna())
znacilke["kategorija_manjka"] = df[kategorija].isna().astype(int)
assert not np.isinf(znacilke["razmerje_agregatov"].to_numpy()).any()
assert znacilke.loc[df[imenovalec].eq(0), "razmerje_agregatov"].isna().all()
print(f"Znano ničelnih imenovalcev: {df[imenovalec].eq(0).sum():,}")
print(f"Neznanih imenovalcev: {df[imenovalec].isna().sum():,}")
print(f"Neznanih razmerij: {znacilke['razmerje_agregatov'].isna().sum():,}")
display(znacilke[[stevec, imenovalec, "razmerje_agregatov", "imenovalec_je_nic"]].head(10))


## 3. Kazalnik: ali je zabeležen stik s podporo?

Pozitivna mera `custcare_Mean` pomeni pozitiven agregat stikov. Kazalnik 0/1 ne
pove števila klicev in ne označuje določenega 30-dnevnega obdobja. Pri manjkajočem
vhodu ohranimo manjkajoč kazalnik. Ne pretvarjamo neznanega stanja v »ne«.

Levi graf pokaže število strank glede na stik s podporo. **Desni graf prikazuje
razmerje iz 2. poglavja**: porabo minut ali prihodke, odvisno od izbranih stolpcev
`stevec` in `imenovalec`. Naslov in oznaka osi se prilagodita tej izbiri.


In [ ]:
stolpec_podpora = "custcare_Mean"
znacilke["stik_s_podporo"] = df[stolpec_podpora].gt(0).astype(float).where(
    df[stolpec_podpora].notna()
)
assert znacilke.loc[df[stolpec_podpora].isna(), "stik_s_podporo"].isna().all()
display(pd.concat([df[[stolpec_podpora]], znacilke[["stik_s_podporo"]]], axis=1).head(10))
display(znacilke["stik_s_podporo"].value_counts(dropna=False).rename("stranke").to_frame())

fig, osi = plt.subplots(1, 2, figsize=(11, 4))
znacilke["stik_s_podporo"].fillna("NEZNANO").value_counts().plot.bar(ax=osi[0])
osi[0].set(xlabel="Stik s podporo (0 = ne, 1 = da)", ylabel="Število strank",
           title="Stranke glede na stik s podporo")
znana_razmerja = znacilke["razmerje_agregatov"].dropna()
osi[1].hist(znana_razmerja, bins=50, edgecolor="white")
naslovi_razmerij = {
    ("avg3mou", "avg6mou"): "Razmerje povprečne porabe minut\n3 meseci / 6 mesecev (2. poglavje)",
    ("avg3rev", "avg6rev"): "Razmerje povprečnih prihodkov\n3 meseci / 6 mesecev (2. poglavje)",
}
naslov_razmerja = naslovi_razmerij.get(
    (stevec, imenovalec), f"Razmerje iz 2. poglavja\n{stevec} / {imenovalec}"
)
osi[1].set(xlabel=f"{stevec} / {imenovalec}", ylabel="Število strank", title=naslov_razmerja)
fig.tight_layout()
plt.show()


### Preveri tudi ničlo in manjkajoč vhod (umetna mini primera)

V dejanskem stolpcu morda ni vseh robnih primerov. Spodnje vrstice so ločen,
jasno označen preizkus pravil, ne sprememba dejanskih strank.


In [ ]:
preizkus = pd.DataFrame({"stevec": [10.0, 10.0, 10.0], "imenovalec": [2.0, 0.0, np.nan],
                        "podpora": [0.0, 2.0, np.nan]})
preizkus["razmerje"] = preizkus["stevec"] / preizkus["imenovalec"].replace(0, np.nan)
preizkus["stik"] = preizkus["podpora"].gt(0).astype(float).where(preizkus["podpora"].notna())
assert preizkus["razmerje"].iloc[1:].isna().all()
assert preizkus["stik"].iloc[:2].tolist() == [0.0, 1.0]
assert pd.isna(preizkus["stik"].iloc[2])
display(preizkus)


## 4. Iz trajanja naročniškega razmerja oblikujemo skupine (dodatno)

Stolpec `months` pove, koliko mesecev je stranka naročnik. Stranke razdelimo
v skupine **do 12 mesecev**, **od 13 do 24 mesecev** in **več kot 24 mesecev**.
Vrstni red teh skupin je jasen, ker izhaja iz trajanja razmerja. Regije pa
obravnavamo kot kategorije brez naravnega vrstnega reda.

Pri razvrščanju izgubimo nekaj podrobnosti: 13 in 24 mesecev na primer spadata
v isto skupino. Izvorni `months` ohrani točno število mesecev; uporabnost skupin
bi morali preveriti pri modeliranju.


In [ ]:
znacilke["razred_trajanja"] = pd.cut(
    df["months"], bins=[0, 12, 24, np.inf], labels=["do 12", "13–24", "nad 24"],
    include_lowest=True, ordered=True,
)
assert not {"Customer_ID", "churn"}.intersection(znacilke.columns)
assert not np.isinf(znacilke.select_dtypes(include="number").to_numpy()).any()
display(znacilke["razred_trajanja"].value_counts(sort=False).rename("stranke").to_frame())
display(pd.DataFrame([
    ["razmerje_agregatov", f"{stevec} / {imenovalec}", "0 ali NaN v imenovalcu → NaN"],
    ["imenovalec_je_nic", f"{imenovalec} == 0", "neznan vhod → NaN"],
    ["stik_s_podporo", f"{stolpec_podpora} > 0", "neznan vhod → NaN"],
    ["kategorija_manjka", f"{kategorija} manjka", "znan kazalnik 0/1"],
], columns=["značilka", "vir / pravilo", "manjkajoče vrednosti"]))


## Kodiranje 1: najprej ločimo učne in testne podatke

V tabeli `znacilke` ena vrstica predstavlja enega kupca oziroma stranko.
Identifikator in morebitno ciljno vrednost hranimo ločeno od značilk.
80 % vrstic namenimo učenju, 20 % pa preverjanju. Ista stranka sme biti samo
v eni od obeh skupin. Ta delitev je namenjena prikazu postopka; način preverjanja
modela bomo podrobneje obravnavali pri modeliranju.

Razmerje uporabe in kazalnik stika s podporo izračunamo po vnaprej določenih pravilih.
Seznam kategorij, njihove pogostosti, mediano za nadomeščanje manjkajočih
vrednosti in parametre skaliranja pa določimo **samo na učnih podatkih**.
Na testnih nato uporabimo iste naučene nastavitve.


In [ ]:
kategoricni_stolpec = kategorija
urejeni_stolpec = 'razred_trajanja'
urejene_kategorije = ['do 12', '13–24', 'nad 24']
numericni_stolpci = ['months', 'razmerje_agregatov', 'imenovalec_je_nic', 'stik_s_podporo', 'kategorija_manjka']

assert znacilke.index.is_unique
assert not {"Customer_ID", "customer_unique_id", "churn"}.intersection(znacilke.columns)
ucni_indeksi, testni_indeksi = train_test_split(znacilke.index, test_size=0.2, random_state=42)
X_train = znacilke.loc[ucni_indeksi].copy()
X_test = znacilke.loc[testni_indeksi].copy()
assert set(kljuci.loc[ucni_indeksi]).isdisjoint(set(kljuci.loc[testni_indeksi]))
print(f"Učnih strank: {len(X_train):,}; testnih: {len(X_test):,}")


## Kodiranje 2: redke kategorije v OTHER in referenčna kategorija

Regija nima naravnega vrstnega reda. Kategorije z manj kot **3 % učnih vrstic**
združimo v `OTHER`. Prag je izbran za prikaz; pogostosti ocenimo samo na učni
množici. Spremeni `prag_redkosti` in preveri, katere kategorije se združijo.

`OneHotEncoder(min_frequency=..., drop="first")` po združevanju izpusti prvo
ohranjeno kategorijo. Ta je **referenčna** in ima v one-hot bloku same ničle.
S tem odpravimo popolno kolinearnost tega bloka s konstantnim členom pri linearni
regresiji; izbiro kodiranja prilagodimo modelu.

`handle_unknown="infrequent_if_exist"` nove vrednosti usmeri v obstoječi `OTHER`.
Scikit-learn skupino interno imenuje `infrequent_sklearn`; v imenih izhodnih
stolpcev jo zaradi berljivosti prikažemo kot `OTHER`.

Manjkajoče vrednosti pred kodiranjem označimo `NEZNANO`. Če je ta oznaka redka ali
še nevidena, gre prav tako v `OTHER`; ločen `kategorija_manjka` ohrani razliko.
ID-jev in ciljnih oznak ne kodiramo kot vhode.


In [ ]:
prag_redkosti = 0.03  # Manj kot 3 % učnih vrstic -> OTHER.


def poimenuj_onehot(stolpec, kategorija):
    oznaka = "OTHER" if kategorija == "infrequent_sklearn" else kategorija
    return f"{stolpec}_{oznaka}"


kat_train = X_train[[kategoricni_stolpec]].astype("object").fillna("NEZNANO")
kat_test = X_test[[kategoricni_stolpec]].astype("object").fillna("NEZNANO")
enkoder = OneHotEncoder(
    min_frequency=prag_redkosti,
    drop="first",
    handle_unknown="infrequent_if_exist",
    sparse_output=False,
    feature_name_combiner=poimenuj_onehot,
)
ucni_onehot = enkoder.fit_transform(kat_train)
testni_onehot = enkoder.transform(kat_test)

redke = enkoder.infrequent_categories_[0]
redke_kategorije = [] if redke is None else redke.tolist()
referencna_kategorija = enkoder.categories_[0][int(enkoder.drop_idx_[0])]
referencna_skupina = (
    "OTHER" if referencna_kategorija in redke_kategorije else referencna_kategorija
)
pregled_kategorij = (
    kat_train[kategoricni_stolpec].value_counts()
    .rename_axis(kategoricni_stolpec).to_frame("ucnih_primerov")
)
pregled_kategorij["delez (%)"] = pregled_kategorij["ucnih_primerov"] / len(kat_train) * 100
pregled_kategorij["skupina"] = pregled_kategorij.index.map(
    lambda kategorija: "OTHER" if kategorija in redke_kategorije else kategorija
)
display(pregled_kategorij.round(2))
print("Redke učne kategorije -> OTHER:", redke_kategorije)
print("Referenčna skupina (same ničle):", referencna_skupina)

imena_onehot = enkoder.get_feature_names_out([kategoricni_stolpec])
train_onehot = pd.DataFrame(ucni_onehot, index=X_train.index, columns=imena_onehot)
test_onehot = pd.DataFrame(testni_onehot, index=X_test.index, columns=imena_onehot)
assert train_onehot.columns.equals(test_onehot.columns)
print("Oblika učne in testne matrike:", train_onehot.shape, test_onehot.shape)
display(pd.concat([kat_train.head(5), train_onehot.head(5)], axis=1))


## Kodiranje 3: referenčna, nova in manjkajoča kategorija

Naslednje tri vrstice so **umetni prikazi**, izpeljani iz kopij testnih značilk.
Prva ima referenčno kategorijo, druga novo regijo, tretja manjkajočo vrednost.
Ne dodajamo jih učnim podatkom in enkoderja ne učimo ponovno.

Pri privzetem primeru je `OTHER` nastal iz redkih učnih kategorij. Referenčna
kategorija dobi same ničle, nova pa 1 v stolpcu `OTHER`. Manjkajoč vhod še vedno
označuje tudi `kategorija_manjka = 1`.

**Robni primer:** če nobena učna kategorija ni redka, skupina `OTHER` ne nastane.
Nova vrednost tedaj dobi same ničle in je v tem bloku ne ločimo od reference.
Spodaj preverimo, ali stolpec `OTHER` obstaja; ne predpostavljamo ga pri vsakem
izbranem stolpcu ali pragu. Opozorilo knjižnice ob novi kategoriji je pri tem
preizkusu pričakovano; preverimo dejanski izhod kodiranja.


In [ ]:
nove_vrstice = X_test.head(3).copy()
nove_vrstice.index = [
    "umetna referenčna kategorija", "umetna nova kategorija", "umetna manjkajoča kategorija"
]
nove_vrstice[kategoricni_stolpec] = [referencna_kategorija, "NOVA_REGIJA_ZA_PRIMER", np.nan]
nove_vrstice["kategorija_manjka"] = nove_vrstice[kategoricni_stolpec].isna().astype(int)
novi_kategoriji = nove_vrstice[[kategoricni_stolpec]].astype("object").fillna("NEZNANO")
slovar_pred = enkoder.categories_[0].copy()
novi_onehot = pd.DataFrame(
    enkoder.transform(novi_kategoriji), index=nove_vrstice.index, columns=imena_onehot
)
np.testing.assert_array_equal(enkoder.categories_[0], slovar_pred)
assert novi_onehot.loc["umetna referenčna kategorija"].eq(0).all()

stolpec_other = f"{kategoricni_stolpec}_OTHER"
if stolpec_other in novi_onehot.columns:
    assert novi_onehot.loc["umetna nova kategorija", stolpec_other] == 1
    print("Nova vrednost gre v OTHER; referenčna kategorija ima same ničle.")
else:
    assert novi_onehot.loc["umetna nova kategorija"].eq(0).all()
    print("Ni ločenega stolpca OTHER: nova vrednost ima v tem bloku isto kodo kot referenca.")
display(pd.concat([novi_kategoriji, nove_vrstice[["kategorija_manjka"]], novi_onehot], axis=1))


## Kodiranje 4: kategorije z vrstnim redom pretvorimo v številčne kode

Iz stolpca `months`, ki pove število mesecev naročniškega razmerja, smo
oblikovali tri razrede. Razvrščeni so od krajšega do daljšega razmerja.
Z `OrdinalEncoder` vsakemu razredu dodelimo številčno kodo. Temu pravimo
**ordinalno kodiranje**.

| Trajanje naročniškega razmerja | Razred | Koda |
|---|---|---:|
| Do 12 mesecev | `do 12` | 0 |
| Od 13 do 24 mesecev | `13–24` | 1 |
| Več kot 24 mesecev | `nad 24` | 2 |

Stranka z 18 meseci naročniškega razmerja dobi kodo **1**, ker spada v razred
`13–24`. Koda označuje razred. Dejansko trajanje njenega razmerja je 18 mesecev.
Vrstni red kod določimo s seznamom `urejene_kategorije`.

- `fit_transform()` uporabimo na učnih vrsticah, `transform()` pa na testnih,
  da za oboje velja isto kodiranje.
- Kategorija, ki je ni na našem seznamu, dobi **−1**. Spodaj to pokažemo z
  umetnim primerom. −1 pomeni »nepoznana kategorija«; ne pomeni negativnega
  trajanja razmerja.
- Razlika med kodama 0 in 1 je enaka razliki med 1 in 2. Razliki med skupinami
  strank pa nista nujno enako veliki. Linearni model bi koraka obravnaval enako;
  po potrebi razrede kodiramo z one-hot.

Tukaj vadimo kodiranje. Razred je izpeljan iz `months` in izgubi podatek o točnem
številu mesecev. Pri modeliranju lahko uporabimo kar `months`; v pripravljeni
različici Pipeline spodaj uporabimo ta izvorni stolpec.


In [ ]:
ord_train = X_train[[urejeni_stolpec]].astype("object").fillna("NEZNANO")
ord_test = X_test[[urejeni_stolpec]].astype("object").fillna("NEZNANO")
ordinalni = OrdinalEncoder(
    categories=[urejene_kategorije], handle_unknown="use_encoded_value", unknown_value=-1
)
ucne_kode = ordinalni.fit_transform(ord_train)
testne_kode = ordinalni.transform(ord_test)
display(pd.DataFrame({"razred": ord_train.iloc[:8, 0], "koda": ucne_kode[:8, 0]}))
print("Vrstni red:", ordinalni.categories_[0].tolist())
print("Koda umetne nove stopnje:", ordinalni.transform(
    pd.DataFrame({urejeni_stolpec: ["NOVA_STOPNJA_ZA_PRIMER"]})
)[0, 0])
